In [1]:
from keras.models import load_model

model = load_model("gesture_recognition.keras")

print("Model loaded!")

2026-06-08 02:12:36.226732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model loaded!


In [ ]:
import os, json, cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score

class_names = ['like', 'dislike', 'stop', 'rock', 'peace']
IMG_SIZE = 64

with open("annot-thu.json", "r") as f:
    ann = json.load(f)

my_images = []
my_labels = []

for label_idx, label_name in enumerate(class_names):
    folder = f"Dataset/{label_name}"

    for filename in sorted(os.listdir(folder)):
        if not filename.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        image_id = os.path.splitext(filename)[0]
        path = os.path.join(folder, filename)

        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # bbox format: [x, y, width, height], normalized 0-1
        x, y, bw, bh = ann[image_id]["bboxes"][0]

        x1 = int(x * w)
        y1 = int(y * h)
        x2 = int((x + bw) * w)
        y2 = int((y + bh) * h)

        crop = img[y1:y2, x1:x2]

        crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
        my_images.append(crop)
        my_labels.append(label_idx)

my_images = np.array(my_images).astype("float32") / 255.0
my_labels = np.array(my_labels)

predictions = model.predict(my_images)
predicted_labels = np.argmax(predictions, axis=1)

print("Accuracy:", accuracy_score(my_labels, predicted_labels))

conf_matrix = confusion_matrix(my_labels, predicted_labels, labels=list(range(len(class_names))))

fig = plt.figure(figsize=(8, 8))
ConfusionMatrixDisplay(conf_matrix, display_labels=class_names).plot(ax=plt.gca())
plt.xticks(rotation=90, ha="center")
plt.title("Confusion Matrix on My Cropped Dataset")
plt.savefig("conf-matrix.png", bbox_inches="tight")
plt.show()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
Accuracy: 0.8666666666666667
